In [14]:
import os
import numpy as np
import pandas as pd
from scipy.signal import resample

# Constants
data_root = "./data/"
SAMPLE_RATE = 60  # Hz

# Window sizes (in samples) per activity
window_sizes = {
    "walking": 200,   #
    "running": 200,
    "bus": 200,
    "biking": 200,
    "sitting": 200,   #
    "stairs": 200     #
}
activities = list(window_sizes.keys())

# Sensors and their axis columns
sensors = {
    "Accelerometer.csv": ["Acceleration x (m/s^2)", "Acceleration y (m/s^2)", "Acceleration z (m/s^2)"],
    "Gyroscope.csv":    ["Gyroscope x (rad/s)",    "Gyroscope y (rad/s)",    "Gyroscope z (rad/s)"],
    "Linear Acceleration.csv": ["Linear Acceleration x (m/s^2)", "Linear Acceleration y (m/s^2)", "Linear Acceleration z (m/s^2)"]
}

def load_activity_data(activity):
    """
    Loads and merges all sensor streams for a given activity into a single DataFrame,
    aligned by timestamp and resampled to uniform SAMPLE_RATE.
    """
    dfs = []
    time_index = None
    for fname, axes in sensors.items():
        path = os.path.join(data_root, activity, fname)
        df = pd.read_csv(path, sep=';')
        # assume first column is time in seconds
        t = df.iloc[:, 0].values
        if time_index is None:
            # create uniform time grid
            duration = t[-1] - t[0]
            n_samples = int(duration * SAMPLE_RATE)
            uniform_t = np.linspace(t[0], t[-1], n_samples)
            time_index = uniform_t
        # resample each axis to the uniform grid
        arr = []
        for axis in axes:
            signal = df[axis].dropna().values
            resampled = np.interp(time_index, t, signal)
            arr.append(resampled)
        # shape: (n_channels, n_samples)
        data = np.vstack(arr).T
        cols = [f"{os.path.splitext(fname)[0]}_{ax}" for ax in ['x','y','z']]
        dfs.append(pd.DataFrame(data, columns=cols))
    # merge on index
    merged = pd.concat(dfs, axis=1)
    return merged.values  # shape (n_samples, n_channels)


def make_windows(X, activity):
    """
    Splits multivariate signal X into non-overlapping windows for the given activity.
    Returns X_windows and corresponding labels.
    """
    ws = window_sizes[activity]
    n_samples, n_channels = X.shape
    n_windows = n_samples // ws
    Xw = X[:n_windows * ws].reshape(n_windows, ws, n_channels)
    y = np.full(n_windows, activities.index(activity), dtype=int)
    print(Xw.shape)
    return Xw, y

In [15]:
# Build dataset
X_list, y_list = [], []
for act in activities:
    X_signal = load_activity_data(act)          # (n_samples, n_channels)
    Xw, yw = make_windows(X_signal, act)        # (n_windows, ws, n_channels), (n_windows,)
    X_list.append(Xw)
    y_list.append(yw)

(684, 200, 9)
(799, 200, 9)
(376, 200, 9)
(265, 200, 9)
(537, 200, 9)


In [16]:
X = np.vstack(X_list)  # (total_windows, ws, n_channels)
y = np.concatenate(y_list)

# Shuffle dataset
perm = np.random.RandomState(42).permutation(len(X))
X = X[perm]
y = y[perm]

In [17]:
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import GRU, Dense
# from tensorflow.keras.callbacks import EarlyStopping
# import pandas as pd


# # 2) Three-way split: train/val/test = 60/20/20
# X_trainval, X_test, y_trainval, y_test = train_test_split(
#     X, y, test_size=0.4, random_state=42, stratify=y
# )
# X_train, X_val, y_train, y_val = train_test_split(
#     X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
# )

# # 3) Scale channels
# n_tr, T, C = X_train.shape
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train.reshape(-1, C)).reshape(n_tr, T, C)
# X_val   = scaler.transform  (X_val.reshape(-1, C)).reshape(X_val.shape[0], T, C)
# X_test  = scaler.transform  (X_test.reshape(-1, C)).reshape(X_test.shape[0], T, C)

# n_classes = len(np.unique(y))

# # 4) Grid definition
# units_list  = [32, 64, 128]
# lrs         = [1e-3, 1e-4]
# batch_sizes = [32, 64]

# best_val_acc = 0.0
# best_model   = None
# best_cfg     = None

# # 5) Search loop
# for units in units_list:
#     for lr in lrs:
#         for bs in batch_sizes:
#             # Build model
#             model = Sequential([
#                 GRU(units, input_shape=(T, C)),
#                 Dense(100, activation='relu'),
#                 Dense(n_classes, activation='softmax')
#             ])
#             model.compile(
#                 optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
#                 loss='sparse_categorical_crossentropy',
#                 metrics=['accuracy']
#             )

#             # Train with early stopping
#             es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
#             hist = model.fit(
#                 X_train, y_train,
#                 validation_data=(X_val, y_val),
#                 epochs=20,
#                 batch_size=bs,
#                 callbacks=[es],
#                 verbose=2
#             )

#             # Evaluate on validation
#             val_acc = max(hist.history['val_accuracy'])
#             print(f"units={units}, lr={lr}, bs={bs} → val_acc={val_acc:.3f}")

#             if val_acc > best_val_acc:
#                 best_val_acc = val_acc
#                 best_model   = model
#                 best_cfg     = (units, lr, bs)

# print(f"\nBest config: units={best_cfg[0]}, lr={best_cfg[1]}, bs={best_cfg[2]}, val_acc={best_val_acc:.3f}")

# # 6) Final test evaluation
# test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
# print(f"Test accuracy: {test_acc:.3f}")


In [18]:
import numpy as np
import pandas as pd
import logging
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv1D, Flatten, Dense,
    LSTM, GRU, Input,
    MultiHeadAttention, Add, LayerNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

logger.info(f"Loaded data: X shape={X.shape}, y shape={y.shape}")

# 2) Three-way split: train/val/test = 60%/20%/20%
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
logger.info(
    f"Data splits: train={X_train.shape[0]}, val={X_val.shape[0]}, test={X_test.shape[0]}"
)

# 3) Scale channels (flatten → scale → reshape)
n_tr, T, C = X_train.shape
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, C)).reshape(n_tr, T, C)
X_val   = scaler.transform(X_val.reshape(-1, C)).reshape(X_val.shape[0], T, C)
X_test  = scaler.transform(X_test.reshape(-1, C)).reshape(X_test.shape[0], T, C)
logger.info("Feature scaling applied to all splits")

n_classes = len(np.unique(y))
logger.info(f"Number of classes: {n_classes}")

# 4) Hyperparameter grid
hidden_dims    = [32, 64, 128]
learning_rates = [1e-3, 1e-4]
batch_sizes    = [32, 64]

# Builder functions for each architecture
def build_cnn(hidden_dim, lr):
    model = Sequential([
        Conv1D(hidden_dim, kernel_size=3, activation='relu', input_shape=(T, C)),
        Flatten(),
        Dense(100, activation='relu'),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_lstm(hidden_dim, lr):
    model = Sequential([
        LSTM(hidden_dim, input_shape=(T, C)),
        Dense(100, activation='relu'),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_gru(hidden_dim, lr):
    model = Sequential([
        GRU(hidden_dim, input_shape=(T, C)),
        Dense(100, activation='relu'),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_transformer(hidden_dim, lr):
    inp = Input(shape=(T, C))
    x = Dense(hidden_dim)(inp)
    attn = MultiHeadAttention(num_heads=4, key_dim=hidden_dim)(x, x)
    x = Add()([x, attn])
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(100, activation='relu')(x)
    out = Dense(n_classes, activation='softmax')(x)
    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

builders = {
    '1D-CNN': build_cnn,
    'LSTM': build_lstm,
    'GRU': build_gru,
    'Transformer': build_transformer
}

# 5) Grid search
results = []
best = {name: {'val_acc': 0, 'cfg': None, 'model': None} for name in builders}

for name, build_fn in builders.items():
    for hd in hidden_dims:
        for lr in learning_rates:
            for bs in batch_sizes:
                logger.info(f"Training {name} with hidden_dim={hd}, lr={lr}, batch_size={bs}")
                model = build_fn(hd, lr)
                es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
                hist = model.fit(
                    X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=20,
                    batch_size=bs,
                    callbacks=[es],
                    verbose=0
                )
                val_acc = max(hist.history['val_accuracy'])
                logger.info(f"Finished {name} config: val_acc={val_acc:.3f}")
                results.append({
                    'model': name,
                    'hidden_dim': hd,
                    'learning_rate': lr,
                    'batch_size': bs,
                    'val_accuracy': val_acc
                })
                if val_acc > best[name]['val_acc']:
                    best[name]['val_acc'] = val_acc
                    best[name]['cfg']    = (hd, lr, bs)
                    best[name]['model']  = model

# 6) Report results
results_df = pd.DataFrame(results)
logger.info("Grid search complete. Results: \n%s", results_df)

for name in builders:
    hd, lr, bs = best[name]['cfg']
    val_acc = best[name]['val_acc']
    logger.info(f"Best {name} -> hidden_dim={hd}, lr={lr}, bs={bs}, val_acc={val_acc:.3f}")
    test_loss, test_acc = best[name]['model'].evaluate(X_test, y_test, verbose=0)
    logger.info(f"{name} test accuracy: {test_acc:.3f}")


2025-06-22 13:02:57 INFO Loaded data: X shape=(2661, 200, 9), y shape=(2661,)
2025-06-22 13:02:57 INFO Data splits: train=1197, val=399, test=1065
2025-06-22 13:02:57 INFO Feature scaling applied to all splits
2025-06-22 13:02:57 INFO Number of classes: 5
2025-06-22 13:02:57 INFO Training 1D-CNN with hidden_dim=32, lr=0.001, batch_size=32
/home/troy/VU_HW/mlqs/venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-06-22 13:03:00 INFO Finished 1D-CNN config: val_acc=0.972
2025-06-22 13:03:00 INFO Training 1D-CNN with hidden_dim=32, lr=0.001, batch_size=64
2025-06-22 13:03:02 INFO Finished 1D-CNN config: val_acc=0.980
2025-06-22 13:03:02 INFO Training 1D-CNN with hidden_dim=32, lr=0.0001, batch_size=32
2